<a href="https://colab.research.google.com/github/toddpirtle/Portfolio/blob/main/Loan%20Default%20Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Loan Default Prediction (Machine Learning and Neural Networks)
This project aims to predict loan defaults using historical lending data, addressing challenges of large datasets and numerous features. The objectives include data cleaning, identifying key predictive factors, and developing a deep learning model to estimate loan default risk.

###Import Libraries
This cell imports the essential Python libraries for data manipulation (pandas, numpy), visualization (matplotlib, seaborn), and machine learning (tensorflow).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn import tree
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

###Mount Google Drive
This cell connects the Colab notebook to your Google Drive, allowing the script to access files stored in your Drive folders.

In [2]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


###Load Dataset
This cell defines the file path for the loan dataset and reads the CSV file into a pandas DataFrame named `loan`.

In [3]:
loan_data = '/content/gdrive/MyDrive/loan_data.csv'
loan = pd.read_csv(loan_data)
loan

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,456251,0,Cash loans,M,N,N,0,157500.0,254700.0,27558.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
307507,456252,0,Cash loans,F,N,Y,0,72000.0,269550.0,12001.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
307508,456253,0,Cash loans,F,N,Y,0,153000.0,677664.0,29979.0,...,0,0,0,0,1.0,0.0,0.0,1.0,0.0,1.0
307509,456254,1,Cash loans,F,N,Y,0,171000.0,370107.0,20205.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


### Create and Preview DataFrame
This cell creates a copy of the loaded data into `df_loan` and displays the first few rows to verify the data structure.

In [4]:
df_loan = pd.DataFrame(loan)
df_loan.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


### Data Quality Check
This cell performs an initial data quality check by displaying the shape, information, and descriptive statistics of the `df_loan` DataFrame.

In [5]:
#Data Quality Check
display(df_loan.shape)
display(df_loan.info())
display(df_loan.describe())

(307511, 122)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 122 entries, SK_ID_CURR to AMT_REQ_CREDIT_BUREAU_YEAR
dtypes: float64(65), int64(41), object(16)
memory usage: 286.2+ MB


None

,SK_ID_CURR,TARGET,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
count,307511.000000,307511.000000,307511.000000,3.075110e+05,3.075110e+05,307499.000000,3.072330e+05,307511.000000,307511.000000,307511.000000,...,307511.000000,307511.000000,307511.000000,307511.000000,265992.000000,265992.000000,265992.000000,265992.000000,265992.000000,265992.000000
mean,278180.518577,0.080729,0.417052,1.687979e+05,5.990260e+05,27108.573909,5.383962e+05,0.020868,-16036.995067,63815.045904,...,0.008130,0.000595,0.000507,0.000335,0.006402,0.007000,0.034362,0.267395,0.265474,1.899974
std,102790.175348,0.272419,0.722121,2.371231e+05,4.024908e+05,14493.737315,3.694465e+05,0.013831,4363.988632,141275.766519,...,0.089798,0.024387,0.022518,0.018299,0.083849,0.110757,0.204685,0.916002,0.794056,1.869295
min,100002.000000,0.000000,0.000000,2.565000e+04,4.500000e+04,1615.500000,4.050000e+04,0.000290,-25229.000000,-17912.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,189145.500000,0.000000,0.000000,1.125000e+05,2.700000e+05,16524.000000,2.385000e+05,0.010006,-19682.000000,-2760.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,278202.000000,0.000000,0.000000,1.471500e+05,5.135310e+05,24903.000000,4.500000e+05,0.018850,-15750.000000,-1213.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
75%,367142.500000,0.000000,1.000000,2.025000e+05,8.086500e+05,34596.000000,6.795000e+05,0.028663,-12413.000000,-289.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.000000
max,456255.000000,1.000000,19.000000,1.170000e+08,4.050000e+06,258025.500000,4.050000e+06,0.072508,-7489.000000,365243.000000,...,1.000000,1.000000,1.000000,1.000000,4.000000,9.000000,8.000000,27.000000,261.000000,25.000000


### Data Preparation: One-Hot Encoding
This cell identifies object-type columns in `df_loan`, performs one-hot encoding on them to convert categorical data into a numerical format, and then displays the head and shape of the new `df_loan_encoded` DataFrame.

In [6]:
object_cols = df_loan.select_dtypes(include='object').columns
print("Object columns to encode:", object_cols)

#One-hot encode object columns
df_loan_encoded = pd.get_dummies(df_loan, columns=object_cols, dummy_na=False)

display(df_loan_encoded.head())
display(df_loan_encoded.shape)

Object columns to encode: Index(['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
       'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE',
       'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE',
       'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE',
       'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE'],
      dtype='object')


,SK_ID_CURR,TARGET,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,...,HOUSETYPE_MODE_terraced house,WALLSMATERIAL_MODE_Block,WALLSMATERIAL_MODE_Mixed,WALLSMATERIAL_MODE_Monolithic,WALLSMATERIAL_MODE_Others,WALLSMATERIAL_MODE_Panel,"WALLSMATERIAL_MODE_Stone, brick",WALLSMATERIAL_MODE_Wooden,EMERGENCYSTATE_MODE_No,EMERGENCYSTATE_MODE_Yes
0,100002,1,0,202500.0,406597.5,24700.5,351000.0,0.018801,-9461,-637,...,False,False,False,False,False,False,True,False,True,False
1,100003,0,0,270000.0,1293502.5,35698.5,1129500.0,0.003541,-16765,-1188,...,False,True,False,False,False,False,False,False,True,False
2,100004,0,0,67500.0,135000.0,6750.0,135000.0,0.010032,-19046,-225,...,False,False,False,False,False,False,False,False,False,False
3,100006,0,0,135000.0,312682.5,29686.5,297000.0,0.008019,-19005,-3039,...,False,False,False,False,False,False,False,False,False,False
4,100007,0,0,121500.0,513000.0,21865.5,513000.0,0.028663,-19932,-3038,...,False,False,False,False,False,False,False,False,False,False


(307511, 246)

### Data Preparation: Impute Missing Values
This cell identifies columns with missing values in `df_loan_encoded` and imputes them with the median value of each respective column to handle missing data.

In [7]:
missing_values_count = df_loan_encoded.isnull().sum()
columns_with_missing_values = missing_values_count[missing_values_count > 0].index.tolist()

print("Columns with missing values:", columns_with_missing_values)
print("\nNumber of columns with missing values:", len(columns_with_missing_values))

#Impute missing values with the median using .loc to avoid SettingWithCopyWarning
for col in columns_with_missing_values:
    if df_loan_encoded[col].dtype in ['float64', 'int64']:
        median_val = df_loan_encoded[col].median()
        df_loan_encoded.loc[:, col] = df_loan_encoded[col].fillna(median_val)

print("\nMissing values after imputation:")
display(df_loan_encoded.isnull().sum().sum())

Columns with missing values: ['AMT_ANNUITY', 'AMT_GOODS_PRICE', 'OWN_CAR_AGE', 'CNT_FAM_MEMBERS', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'ELEVATORS_AVG', 'ENTRANCES_AVG', 'FLOORSMAX_AVG', 'FLOORSMIN_AVG', 'LANDAREA_AVG', 'LIVINGAPARTMENTS_AVG', 'LIVINGAREA_AVG', 'NONLIVINGAPARTMENTS_AVG', 'NONLIVINGAREA_AVG', 'APARTMENTS_MODE', 'BASEMENTAREA_MODE', 'YEARS_BEGINEXPLUATATION_MODE', 'YEARS_BUILD_MODE', 'COMMONAREA_MODE', 'ELEVATORS_MODE', 'ENTRANCES_MODE', 'FLOORSMAX_MODE', 'FLOORSMIN_MODE', 'LANDAREA_MODE', 'LIVINGAPARTMENTS_MODE', 'LIVINGAREA_MODE', 'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAREA_MODE', 'APARTMENTS_MEDI', 'BASEMENTAREA_MEDI', 'YEARS_BEGINEXPLUATATION_MEDI', 'YEARS_BUILD_MEDI', 'COMMONAREA_MEDI', 'ELEVATORS_MEDI', 'ENTRANCES_MEDI', 'FLOORSMAX_MEDI', 'FLOORSMIN_MEDI', 'LANDAREA_MEDI', 'LIVINGAPARTMENTS_MEDI', 'LIVINGAREA_MEDI', 'NONLIVINGAPARTMENTS_MEDI', 'NONLIVI

np.int64(0)

### Data Overview After Encoding and Imputation
This cell displays the shape, data types, and descriptive statistics of the `df_loan_encoded` DataFrame after encoding and imputation.

In [8]:
display(df_loan_encoded.shape)
display(df_loan_encoded.info())
display(df_loan_encoded.describe())

(307511, 246)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 246 entries, SK_ID_CURR to EMERGENCYSTATE_MODE_Yes
dtypes: bool(140), float64(65), int64(41)
memory usage: 289.7 MB


None

,SK_ID_CURR,TARGET,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
count,307511.000000,307511.000000,307511.000000,3.075110e+05,3.075110e+05,307511.000000,3.075110e+05,307511.000000,307511.000000,307511.000000,...,307511.000000,307511.000000,307511.000000,307511.000000,307511.000000,307511.000000,307511.000000,307511.000000,307511.000000,307511.000000
mean,278180.518577,0.080729,0.417052,1.687979e+05,5.990260e+05,27108.487841,5.383163e+05,0.020868,-16036.995067,63815.045904,...,0.008130,0.000595,0.000507,0.000335,0.005538,0.006055,0.029723,0.231293,0.229631,1.778463
std,102790.175348,0.272419,0.722121,2.371231e+05,4.024908e+05,14493.461065,3.692890e+05,0.013831,4363.988632,141275.766519,...,0.089798,0.024387,0.022518,0.018299,0.078014,0.103037,0.190728,0.856810,0.744059,1.765523
min,100002.000000,0.000000,0.000000,2.565000e+04,4.500000e+04,1615.500000,4.050000e+04,0.000290,-25229.000000,-17912.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,189145.500000,0.000000,0.000000,1.125000e+05,2.700000e+05,16524.000000,2.385000e+05,0.010006,-19682.000000,-2760.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
50%,278202.000000,0.000000,0.000000,1.471500e+05,5.135310e+05,24903.000000,4.500000e+05,0.018850,-15750.000000,-1213.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
75%,367142.500000,0.000000,1.000000,2.025000e+05,8.086500e+05,34596.000000,6.795000e+05,0.028663,-12413.000000,-289.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.000000
max,456255.000000,1.000000,19.000000,1.170000e+08,4.050000e+06,258025.500000,4.050000e+06,0.072508,-7489.000000,365243.000000,...,1.000000,1.000000,1.000000,1.000000,4.000000,9.000000,8.000000,27.000000,261.000000,25.000000


### Data Preparation: SMOTE for Class Balancing
This cell separates the features (X) and the target variable (y) from `df_loan_encoded` and then applies the SMOTE technique to balance the dataset, addressing class imbalance.

In [9]:
#Separate features (X) and target (y)
X = df_loan_encoded.drop('TARGET', axis=1)
y = df_loan_encoded['TARGET']

#Apply SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

display(X_resampled.shape)
display(y_resampled.value_counts())

(565372, 245)

,count
TARGET,
1,282686
0,282686


### Feature Engineering: Identify Columns for Scaling
This cell identifies numerical columns that require scaling and boolean columns that do not, preparing for the scaling process.

In [10]:
numerical_cols = df_loan_encoded.select_dtypes(include=['float64', 'int64']).columns

#Exclude the target variable if it's present in the numerical columns
if 'TARGET' in numerical_cols:
    numerical_cols = numerical_cols.drop('TARGET')
print("Columns that need scaling:")
display(numerical_cols)

#Identify columns that do not need scaling (boolean columns)
boolean_cols = df_loan_encoded.select_dtypes(include='bool').columns
print("\nColumns that do not need scaling:")
display(boolean_cols)

Columns that need scaling:


Index(['SK_ID_CURR', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT',
       'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'REGION_POPULATION_RELATIVE',
       'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION',
       ...
       'FLAG_DOCUMENT_18', 'FLAG_DOCUMENT_19', 'FLAG_DOCUMENT_20',
       'FLAG_DOCUMENT_21', 'AMT_REQ_CREDIT_BUREAU_HOUR',
       'AMT_REQ_CREDIT_BUREAU_DAY', 'AMT_REQ_CREDIT_BUREAU_WEEK',
       'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT',
       'AMT_REQ_CREDIT_BUREAU_YEAR'],
      dtype='object', length=105)


Columns that do not need scaling:


Index(['NAME_CONTRACT_TYPE_Cash loans', 'NAME_CONTRACT_TYPE_Revolving loans',
       'CODE_GENDER_F', 'CODE_GENDER_M', 'CODE_GENDER_XNA', 'FLAG_OWN_CAR_N',
       'FLAG_OWN_CAR_Y', 'FLAG_OWN_REALTY_N', 'FLAG_OWN_REALTY_Y',
       'NAME_TYPE_SUITE_Children',
       ...
       'HOUSETYPE_MODE_terraced house', 'WALLSMATERIAL_MODE_Block',
       'WALLSMATERIAL_MODE_Mixed', 'WALLSMATERIAL_MODE_Monolithic',
       'WALLSMATERIAL_MODE_Others', 'WALLSMATERIAL_MODE_Panel',
       'WALLSMATERIAL_MODE_Stone, brick', 'WALLSMATERIAL_MODE_Wooden',
       'EMERGENCYSTATE_MODE_No', 'EMERGENCYSTATE_MODE_Yes'],
      dtype='object', length=140)

### Feature Engineering: Scale Numerical Features
This cell scales the identified numerical columns using `StandardScaler` and then combines them with the original boolean columns into a new DataFrame, `df_loan_processed`.

In [11]:
#Initialize the StandardScaler
scaler = StandardScaler()

#Apply the scaler to the numerical columns
df_loan_scaled_numerical = scaler.fit_transform(df_loan_encoded[numerical_cols])

#Create a new DataFrame with scaled numerical columns
df_loan_scaled_numerical = pd.DataFrame(df_loan_scaled_numerical, columns=numerical_cols)

#Combine the scaled numerical columns with the boolean columns
df_loan_processed = pd.concat([df_loan_scaled_numerical, df_loan_encoded[boolean_cols]], axis=1)

print("DataFrame with scaled numerical columns and original boolean columns:")
display(df_loan_processed.head())
display(df_loan_processed.shape)

DataFrame with scaled numerical columns and original boolean columns:


,SK_ID_CURR,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,...,HOUSETYPE_MODE_terraced house,WALLSMATERIAL_MODE_Block,WALLSMATERIAL_MODE_Mixed,WALLSMATERIAL_MODE_Monolithic,WALLSMATERIAL_MODE_Others,WALLSMATERIAL_MODE_Panel,"WALLSMATERIAL_MODE_Stone, brick",WALLSMATERIAL_MODE_Wooden,EMERGENCYSTATE_MODE_No,EMERGENCYSTATE_MODE_Yes
0,-1.733423,-0.577538,0.142129,-0.478095,-0.166143,-0.507236,-0.149452,1.506880,-0.456215,0.379837,...,False,False,False,False,False,False,True,False,True,False
1,-1.733413,-0.577538,0.426792,1.725450,0.592683,1.600873,-1.252750,-0.166821,-0.460115,1.078697,...,False,True,False,False,False,False,False,False,True,False
2,-1.733403,-0.577538,-0.427196,-1.152888,-1.404669,-1.092145,-0.783451,-0.689509,-0.453299,0.206116,...,False,False,False,False,False,False,False,False,False,False
3,-1.733384,-0.577538,-0.142533,-0.711430,0.177874,-0.653463,-0.928991,-0.680114,-0.473217,-1.375829,...,False,False,False,False,False,False,False,False,False,False
4,-1.733374,-0.577538,-0.199466,-0.213734,-0.361749,-0.068554,0.563570,-0.892535,-0.473210,0.191639,...,False,False,False,False,False,False,False,False,False,False


(307511, 245)

### Data Splitting
This cell splits the resampled dataset (`X_resampled`, `y_resampled`) into training and testing sets for both features and the target variable, ensuring a stratified split.

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, train_size=0.8, test_size=0.2, random_state=42, stratify=y_resampled)

display(X_train.shape)
display(X_test.shape)
display(y_train.shape)
display(y_test.shape)

(452297, 245)

(113075, 245)

(452297,)

(113075,)

### Scale Training and Testing Data
This cell scales the training and testing feature sets (`X_train`, `X_test`) using `StandardScaler`, fitting the scaler on the training data and then transforming both sets.

In [13]:
#Initialize the StandardScaler
scaler = StandardScaler()

#Fit the scaler on the training data and transform both training and testing data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

display(X_train_scaled.shape)
display(X_test_scaled.shape)

(452297, 245)

(113075, 245)

### Model Training and Selection: Artificial Neural Network (ANN) Definition
This cell defines and compiles an Artificial Neural Network (ANN) model using TensorFlow/Keras, specifying its architecture with three dense hidden layers and a sigmoid output layer.

In [14]:
#Define the model
model = Sequential()

#Set the first hidden layer
model.add(Dense(units=12, activation='relu', input_shape=(X_train_scaled.shape[1],)))

#Add a second hidden layer
model.add(Dense(units=16, activation='relu'))

#Add a third hidden layer
model.add(Dense(units=8, activation='relu'))

#Set the output layer using a sigmoid function
model.add(Dense(units=1, activation='sigmoid'))

#Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

#Print the model summary
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 12)             │         2,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,305 (12.91 KB)

 Trainable params: 3,305 (12.91 KB)

 Non-trainable params: 0 (0.00 B)

### Model Training: Artificial Neural Network
This cell trains the defined Artificial Neural Network model using the scaled training data, specifying epochs, batch size, and a validation split.

In [15]:
model.fit(X_train_scaled, y_train, epochs=25, batch_size=32, validation_split=0.2, verbose=1)

Epoch 1/25
11308/11308 ━━━━━━━━━━━━━━━━━━━━ 24s 2ms/step - accuracy: 0.9489 - loss: 0.1551 - val_accuracy: 0.9550 - val_loss: 0.1403
Epoch 2/25
11308/11308 ━━━━━━━━━━━━━━━━━━━━ 22s 2ms/step - accuracy: 0.9548 - loss: 0.1407 - val_accuracy: 0.9556 - val_loss: 0.1391
Epoch 3/25
11308/11308 ━━━━━━━━━━━━━━━━━━━━ 23s 2ms/step - accuracy: 0.9550 - loss: 0.1397 - val_accuracy: 0.9553 - val_loss: 0.1390
Epoch 4/25
11308/11308 ━━━━━━━━━━━━━━━━━━━━ 24s 2ms/step - accuracy: 0.9551 - loss: 0.1391 - val_accuracy: 0.9558 - val_loss: 0.1379
Epoch 5/25
11308/11308 ━━━━━━━━━━━━━━━━━━━━ 28s 2ms/step - accuracy: 0.9553 - loss: 0.1387 - val_accuracy: 0.9558 - val_loss: 0.1376
Epoch 6/25
11308/11308 ━━━━━━━━━━━━━━━━━━━━ 24s 2ms/step - accuracy: 0.9553 - loss: 0.1381 - val_accuracy: 0.9558 - val_loss: 0.1384
Epoch 7/25
11308/11308 ━━━━━━━━━━━━━━━━━━━━ 23s 2ms/step - accuracy: 0.9553 - loss: 0.1380 - val_accuracy: 0.9555 - val_loss: 0.1383
Epoch 8/25
11308/11308 ━━━━━━━━━━━━━━━━━━━━ 23s 2ms/step - accuracy: 

### Model Evaluation Function
This cell defines a function `evaluate_model` to calculate and print common classification metrics (accuracy, precision, recall, F1 score) and the confusion matrix for a given model's predictions.

In [16]:
def evaluate_model(model_name, y_test, y_hat):
    from sklearn import metrics

    acc = metrics.accuracy_score(y_test, y_hat)
    prec = metrics.precision_score(y_test, y_hat)
    rec = metrics.recall_score(y_test, y_hat)
    f1 = metrics.f1_score(y_test, y_hat)

    conf_matrix = metrics.confusion_matrix(y_test, y_hat)
    print("\n"*2)
    print("**"*5)
    print(model_name)
    print("**"*5)

    print(f"Accuracy: {acc}")
    print(f"Precision: {prec}")
    print(f"Recall: {rec}")
    print(f"F1 Score: {f1}")
    print("Confusion Matrix: \n", conf_matrix)

    return {"acc": acc, "prec": prec, "recall": rec, "f1 score": f1, "conf matrix": conf_matrix}

### ANN Predictions
This cell uses the trained Artificial Neural Network model to make predictions on the scaled test data (`X_test_scaled`).

In [17]:
y_pred= model.predict(X_test_scaled)
y_pred

3534/3534 ━━━━━━━━━━━━━━━━━━━━ 3s 909us/step


array([[0.9999929 ],
       [0.056069  ],
       [1.        ],
       ...,
       [0.04181581],
       [1.        ],
       [1.        ]], dtype=float32)

### Convert ANN Predictions to Binary
This cell converts the probabilistic predictions from the Artificial Neural Network into binary class labels (0 or 1) using a threshold of 0.5.

In [18]:
y_hat = (y_pred > 0.5).astype(int)
y_hat

array([[1],
       [0],
       [1],
       ...,
       [0],
       [1],
       [1]])

### Evaluate Artificial Neural Network
This cell evaluates the performance of the Artificial Neural Network model using the `evaluate_model` function with the true test labels and the binary predictions.

In [19]:
evaluate_model(model_name="Artificial Neural Network", y_test=y_test, y_hat=y_hat)




**********
Artificial Neural Network
**********
Accuracy: 0.9553040017687375
Precision: 0.9954003964511845
Recall: 0.9148345331375913
F1 Score: 0.9534184961934782
Confusion Matrix: 
 [[56299   239]
 [ 4815 51722]]


{'acc': 0.9553040017687375,
 'prec': 0.9954003964511845,
 'recall': 0.9148345331375913,
 'f1 score': 0.9534184961934782,
 'conf matrix': array([[56299,   239],
        [ 4815, 51722]])}

### Model Training and Selection: Decision Tree Classifier
This cell initializes, trains, and evaluates a Decision Tree Classifier model using the scaled training and testing data.

In [20]:
dtc = tree.DecisionTreeClassifier(random_state=42)
dtc.fit(X_train_scaled, y_train)
dtc_evaluation = evaluate_model(model_name="Decision Tree Classifier", y_test=y_test, y_hat=dtc.predict(X_test_scaled))




**********
Decision Tree Classifier
**********
Accuracy: 0.9080433340703074
Precision: 0.9014949790285247
Recall: 0.9161964731061075
F1 Score: 0.9087862732025686
Confusion Matrix: 
 [[50878  5660]
 [ 4738 51799]]


### Model Training and Selection: Random Forest Classifier
This cell initializes, trains, and evaluates a Random Forest Classifier model using the scaled training and testing data.

In [21]:
def evaluate_model(model_name, y_test, y_hat):
    acc = metrics.accuracy_score(y_test, y_hat)
    prec = metrics.precision_score(y_test, y_hat)
    rec = metrics.recall_score(y_test, y_hat)
    f1 = metrics.f1_score(y_test, y_hat)

    conf_matrix = metrics.confusion_matrix(y_test, y_hat)
    print("\n"*2)
    print("**"*5)
    print(model_name)
    print("**"*5)

    print(f"Accuracy: {acc}")
    print(f"Precision: {prec}")
    print(f"Recall: {rec}")
    print(f"F1 Score: {f1}")
    print("Confusion Matrix: \n", conf_matrix)

    return {"acc": acc, "prec": prec, "recall": rec, "f1 score": f1, "conf matrix": conf_matrix}


rfc = RandomForestClassifier(random_state=42)
rfc.fit(X_train_scaled, y_train)
rfc_evaluation = evaluate_model(model_name="Random Forest Classifier", y_test=y_test, y_hat=rfc.predict(X_test_scaled))

#Initialize the Random Forest classifier, train the model, make predictions, evaluate the model
rf_model = RandomForestClassifier(random_state=42)




**********
Random Forest Classifier
**********
Accuracy: 0.9557284987839929
Precision: 0.998317377429649
Recall: 0.9129950298034916
F1 Score: 0.9537517784224239
Confusion Matrix: 
 [[56451    87]
 [ 4919 51618]]


### Model Training and Selection: Linear Support Vector Machine (LinearSVC)
This cell initializes, trains, and evaluates a Linear Support Vector Machine (LinearSVC) model using the scaled training and testing data.

In [22]:
linear_svm_model = LinearSVC(random_state=42)
linear_svm_model.fit(X_train_scaled, y_train)
linear_svm_predictions = linear_svm_model.predict(X_test_scaled)
linear_svm_evaluation = evaluate_model(model_name="Linear Support Vector Machine", y_test=y_test, y_hat=linear_svm_predictions)




**********
Linear Support Vector Machine
**********
Accuracy: 0.9556842803449038
Precision: 1.0
Recall: 0.9113677768540955
F1 Score: 0.9536289016592173
Confusion Matrix: 
 [[56538     0]
 [ 5011 51526]]


### Final Model Training: Linear SVM
This cell re-initializes and retrains the Linear Support Vector Machine model on the entire training dataset (`X_train_scaled`, `y_train`) for final deployment.

In [23]:
final_linear_svm_model = LinearSVC(random_state=42)

#Train the model on the entire training dataset
final_linear_svm_model.fit(X_train_scaled, y_train)

print("Linear SVM model retrained on the entire training dataset.")

Linear SVM model retrained on the entire training dataset.


### Model Testing: Final Linear SVM
This cell makes predictions with the final Linear Support Vector Machine model on the test set, evaluates its performance using the `evaluate_model` function, and prints a detailed classification report.

In [24]:
#Final Model Testing

#Make predictions on the test set
final_linear_svm_predictions = final_linear_svm_model.predict(X_test_scaled)

#Evaluate the model
final_linear_svm_evaluation = evaluate_model(model_name="Final Linear Support Vector Machine", y_test=y_test, y_hat=final_linear_svm_predictions)

#Generate and print the classification report
print("\nClassification Report:")
print(classification_report(y_test, final_linear_svm_predictions))




**********
Final Linear Support Vector Machine
**********
Accuracy: 0.9556842803449038
Precision: 1.0
Recall: 0.9113677768540955
F1 Score: 0.9536289016592173
Confusion Matrix: 
 [[56538     0]
 [ 5011 51526]]

Classification Report:
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     56538
           1       1.00      0.91      0.95     56537

    accuracy                           0.96    113075
   macro avg       0.96      0.96      0.96    113075
weighted avg       0.96      0.96      0.96    113075



## Final Model Performance Summary and Findings

In this project, we evaluated four distinct machine learning architectures to predict loan defaults: **Artificial Neural Networks (ANN)**, **Decision Tree Classifiers**, **Random Forest Classifiers**, and **Linear Support Vector Machines (LinearSVC)**. After rigorous data preprocessing, including SMOTE for class balancing and standard scaling, the following results were observed:

### Model Comparison Metrics

| Model | Accuracy | Precision | Recall | F1 Score |
| :--- | :---: | :---: | :---: | :---: |
| **Artificial Neural Network** | 0.9553 | 0.9954 | 0.9148 | 0.9534 |
| **Decision Tree Classifier** | 0.9080 | 0.9015 | 0.9162 | 0.9088 |
| **Random Forest Classifier** | 0.9557 | 0.9983 | 0.9130 | 0.9538 |
| **Linear Support Vector Machine** | 0.9557 | 1.0000 | 0.9114 | 0.9536 |

### Key Findings and Final Selection

1.  **Top Performers:** The Random Forest and Linear SVM models achieved nearly identical accuracy and F1 scores, significantly outperforming the Decision Tree.
2.  **Precision Excellence:** The **Linear Support Vector Machine** achieved a perfect precision score (1.00), effectively eliminating false positives in the test set.
3.  **Rationale:** In the context of loan default prediction, the cost of incorrectly flagging a reliable borrower as a 'defaulter' (False Positive) can be high for business growth. Because the Linear SVM provides maximum reliability in its positive predictions while maintaining a high recall (91.1%), it was selected as the optimal production model.

### Conclusion
The final Linear SVM model demonstrated robust generalization capabilities with an overall accuracy of **95.57%**. This model is well-suited for deployment in scenarios where high-confidence default identification is required.